# CUDA at Scale Assignment: Batch Image Processing with CUDA NPP
This Jupyter Notebook provides the complete workflow to compile, execute, and evaluate the CUDA NPP Batch Image Processing assignment on Google Colab (NVIDIA T4 GPU, Compute Capability `sm_75`).

### Pipeline Steps:
1. **GPU & CUDA Environment Verification**
2. **Dependency Installation (`libopencv-dev`)**
3. **Repository Clone / Workspace Navigation**
4. **Compilation via NVCC and Makefile (`-arch=sm_75`)**
5. **Dataset Population (USC SIPI Image Database)**
6. **Pipeline Execution (`run.sh`) & Log Artifact Generation**
7. **Visual Artifact Inspection (Before vs. After)**
8. **Export Execution Proof Artifacts (`output.zip`)**

## Step 1: Verify Hardware & CUDA Environment
Ensure that a GPU runtime (preferably NVIDIA T4) is allocated.

In [ ]:
!nvidia-smi

## Step 2: Verify CUDA Toolkit Version
Confirm that NVCC is accessible in `/usr/local/cuda`.

In [ ]:
!nvcc --version

## Step 3: Install OpenCV Development Libraries
Install `libopencv-dev` using `apt-get` to enable C++ header `<opencv2/opencv.hpp>` and `pkg-config` linkage.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y libopencv-dev

## Step 4: Clone Repository or Setup Project Workspace
Clone your public code repository, or if you uploaded project files directly to Colab, change directory into the workspace.

In [ ]:
# Replace <MY_REPO_URL> with your actual GitHub repository URL
# !git clone <MY_REPO_URL>
# %cd <repo-name>

# If running in current working directory:
%pwd

## Step 5: Build CUDA Project via Makefile
Compile the CUDA C++ application targeting `sm_75` to generate `bin/processor`.

In [ ]:
%%bash
make clean
make all

## Step 6: Populate Input Dataset (`./data`)
Upload sample images from USC SIPI database or download sample images into `./data`.

In [ ]:
import os
from google.colab import files

os.makedirs('./data', exist_ok=True)

# Option A: Interactively upload files from local machine
# print("Please select SIPI dataset image files to upload:")
# uploaded = files.upload()
# for filename in uploaded.keys():
#     os.rename(filename, os.path.join('./data', filename))

# Option B: Download standard sample SIPI dataset via wget
!wget -q -O ./data/4.1.01.tiff http://sipi.usc.edu/database/download.php?vol=misc\&img=4.1.01
!wget -q -O ./data/4.1.02.tiff http://sipi.usc.edu/database/download.php?vol=misc\&img=4.1.02
!wget -q -O ./data/4.1.03.tiff http://sipi.usc.edu/database/download.php?vol=misc\&img=4.1.03
!wget -q -O ./data/4.1.04.tiff http://sipi.usc.edu/database/download.php?vol=misc\&img=4.1.04
!wget -q -O ./data/4.1.05.tiff http://sipi.usc.edu/database/download.php?vol=misc\&img=4.1.05

print("Data folder contents:", os.listdir('./data'))

## Step 7: Run Batch Processing Pipeline
Execute `run.sh` driver script to run GPU NPP filters, CPU baseline benchmarks, and capture logs.

In [ ]:
!bash run.sh

## Step 8: Visual Artifact Inspection (Before / After Comparison)
Display original input image alongside CUDA NPP Gaussian Blur and Sobel Filter outputs.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import os

data_files = [f for f in os.listdir('./data') if f.endswith(('.tiff', '.png', '.jpg'))]
if data_files:
    sample_file = data_files[0]
    orig_path = os.path.join('./data', sample_file)
    gauss_path = os.path.join('./output', sample_file)
    sobel_path = os.path.join('./output/sobel', sample_file)

    img_orig = cv2.imread(orig_path)
    img_gauss = cv2.imread(gauss_path)
    img_sobel = cv2.imread(sobel_path)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    if img_orig is not None:
        axes[0].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f"Original ({sample_file})")
        axes[0].axis('off')

    if img_gauss is not None:
        axes[1].imshow(cv2.cvtColor(img_gauss, cv2.COLOR_BGR2RGB))
        axes[1].set_title("CUDA NPP Gaussian Blur")
        axes[1].axis('off')

    if img_sobel is not None:
        axes[2].imshow(cv2.cvtColor(img_sobel, cv2.COLOR_BGR2RGB))
        axes[2].set_title("CUDA NPP Sobel Edge")
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()

## Step 9: Download Execution Proof & Artifact Zip
Compress `./output` folder containing `log.txt` and processed images, then trigger local download.

In [ ]:
!zip -r output_artifacts.zip output/
from google.colab import files
files.download('output_artifacts.zip')